# Chapter 7: Topic Modeling with Neural Topic Modeling and Latent Dirichlet Allocation

This notebook demonstrates topic modeling using two approaches:
1. **Latent Dirichlet Allocation (LDA)** on Amazon SageMaker
2. **Neural Topic Modeling (NTM)** on Amazon SageMaker

We use Star Trek episode scripts as our corpus to discover latent topics.

In [ ]:
!pip install "sagemaker>=2.0,<3.0"

In [ ]:
!pip show sagemaker

---
## Part 1 – LDA on SageMaker

Latent Dirichlet Allocation (LDA) is a generative probabilistic model that discovers
abstract topics in a collection of documents. Amazon SageMaker provides a built-in LDA
algorithm optimized for large-scale training.

### 1. Load Dataset

Load the Star Trek scripts dataset from a JSON file.

In [ ]:
import pandas as pd
scripts = pd.read_json('all_scripts_raw.json')
print(scripts.shape)
scripts.head()

### 2. Pre-process with NLTK

We use NLTK for tokenization and stopword removal. The `clean` function
removes short lines (likely stage directions or formatting artifacts).

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def clean(ep_text):
    sentence_list = ep_text.split('\n')
    sentence_list = [line for line in sentence_list if len(line) > 5]
    return ' '.join(sentence_list)

### 3. Prepare Episode 1 (TNG)

Extract the first episode from The Next Generation (TNG), split into documents
(one per line), tokenize, and remove stopwords and short tokens.

In [ ]:
ep_1 = scripts.iloc[0]['TNG']
documents = ep_1.split('\n')
documents = [doc for doc in documents if len(doc) > 5]
print(len(documents))
doc_tokens = []
for doc in documents:
    tokens = nltk.word_tokenize(doc)
    tokens = [t for t in tokens if t.lower() not in stop_words]
    tokens = [t for t in tokens if len(t) > 1]
    doc_tokens.append(tokens)

### 4. Build Document-Word Matrix

Construct a vocabulary from all tokens and build a document-term matrix
where each row is a document and each column is a word count.

In [ ]:
word_list = []
for doc_token in doc_tokens:
    for word in doc_token:
        if word != '' and word not in word_list:
            word_list.append(word)
word_list = list(set(word_list))
print(f"There are {len(word_list)} unique words in the corpus.")

from collections import Counter
final_df = []
for doc_token in doc_tokens:
    doc_row = {}
    tmp_dict = Counter(doc_token)
    for word in word_list:
        doc_row[word] = tmp_dict.get(word, 0)
    final_df.append(doc_row)
lda_df = pd.DataFrame(final_df)
print(lda_df.shape)
lda_df.head()

### 5. SageMaker LDA Training Job

Upload the document-term matrix to S3 and configure an LDA training job
using the SageMaker built-in LDA algorithm container.

In [ ]:
import boto3
import sagemaker
role = sagemaker.get_execution_role()
sm_session = sagemaker.session.Session()
bucket = sm_session.default_bucket()
print(role, bucket)
input_uri = f"s3://{bucket}/lda/input.csv"
lda_df.to_csv(input_uri, header=None, index=False)

from sagemaker.amazon.amazon_estimator import get_image_uri
region_name = boto3.Session().region_name
container = get_image_uri(region_name, "lda")
print("Using SageMaker LDA container: {} ({})".format(container, region_name))
num_topics = 5
minibatch_size = 100
vocab_size = len(word_list)
lda = sagemaker.estimator.Estimator(
    container,
    role,
    output_path="s3://{}/{}/output".format(bucket, "lda"),
    instance_count=1,
    instance_type="ml.c4.2xlarge",
    sagemaker_session=sm_session,
)
lda.set_hyperparameters(
    num_topics=num_topics,
    feature_dim=vocab_size,
    mini_batch_size=minibatch_size,
    alpha0=1.0,
)
from sagemaker.inputs import TrainingInput
train_input = TrainingInput(
    s3_data=input_uri,
    content_type='text/csv;label_size=0'
)
lda.fit({"train": train_input})

### 6. Deploy LDA and Generate Predictions

Deploy the trained LDA model to an endpoint and generate topic mixture
predictions for the first 10 documents.

In [ ]:
lda_inference = lda.deploy(
    initial_instance_count=1,
    instance_type="ml.m4.xlarge",
)
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import JSONDeserializer

lda_inference.serializer = CSVSerializer()
lda_inference.deserializer = JSONDeserializer()
results = lda_inference.predict(lda_df[:10].to_numpy())
results_df = pd.DataFrame(results['predictions'])
results_df = results_df['topic_mixture'].astype(str).str.split(',', expand=True)
results_df.columns = ['Topic1', 'Topic2', 'Topic3', 'Topic4', 'Topic5']
results_df

### 7. Post-process Predictions and Show Majority Topic

Clean up bracket characters from the topic weights, convert to numeric,
and determine the majority (highest-weight) topic for each document.

In [ ]:
def remove_brackets(x):
    if isinstance(x, str):
        return x.removeprefix('[').removesuffix(']')
    return x

In [ ]:
results_df['Topic1'] = results_df['Topic1'].apply(lambda x: remove_brackets(x))
results_df['Topic5'] = results_df['Topic5'].apply(lambda x: remove_brackets(x))
results_df = results_df.apply(pd.to_numeric, axis=1)

In [ ]:
results_df['MajorityTopic'] = results_df.apply(lambda x: x.argmax(), axis=1)
results_df['MajorityTopic'] = results_df['MajorityTopic'] + 1
results_df['InputText'] = documents[:10]
results_df[['InputText', 'MajorityTopic']]

### 8. Download and Inspect Trained LDA Model

Retrieve the model artifact from S3, extract it, and examine the files.

In [ ]:
training_job_name = lda.latest_training_job.job_name
sm_client = boto3.client("sagemaker")
response = sm_client.describe_training_job(TrainingJobName=training_job_name)
model_uri = response['ModelArtifacts']['S3ModelArtifacts']
print("Model location is:", model_uri)
sagemaker.s3.S3Downloader.download(model_uri, "model")
%cd model
!tar xvf model.tar.gz
import os
for file in os.listdir('.'):
    if 'tar.gz' not in file:
        print("Model file found:", file)

### 9. Load and View Alpha and Beta Parameters

The LDA model produces two key parameter sets:
- **Alpha**: Topic prior distribution (shape: num_topics)
- **Beta**: Topic-word distribution matrix (shape: num_topics × vocab_size)

In [ ]:
!pip install mxnet --quiet

import numpy as np
np.bool = bool  # Compatibility fix for older MXNet

import mxnet as mx

alpha, beta = mx.ndarray.load('model_algo-1')
alpha = alpha.asnumpy()
beta = beta.asnumpy()

In [ ]:
print("Alpha shape:", alpha.shape)
print("Beta shape:", beta.shape)

### 10. Cleanup LDA Endpoint

Delete the LDA inference endpoint to avoid ongoing charges.

In [ ]:
lda_inference.delete_endpoint()
print("LDA endpoint deleted.")

---
## Part 2 – Neural Topic Modeling (NTM) on SageMaker

Neural Topic Model (NTM) is a neural network-based approach to topic modeling.
It uses a variational autoencoder architecture to learn topic representations.
NTM can capture more complex patterns than traditional LDA.

### 11. NTM Training Job

Configure and launch an NTM training job using the same input data prepared for LDA.

In [ ]:
container = get_image_uri(region_name, "ntm")
print("Using SageMaker NTM container: {} ({})".format(container, region_name))
num_topics = 5
minibatch_size = 100
vocab_size = len(word_list)
ntm = sagemaker.estimator.Estimator(
    container,
    role,
    output_path="s3://{}/{}/output".format(bucket, "ntm"),
    instance_count=1,
    instance_type="ml.c4.2xlarge",
    sagemaker_session=sm_session,
)
ntm.set_hyperparameters(
    num_topics=num_topics,
    feature_dim=vocab_size,
    epochs=10,
    learning_rate=0.001,
    optimizer='sgd'
)
from sagemaker.inputs import TrainingInput
train_input = TrainingInput(
    s3_data=input_uri,
    content_type='text/csv;label_size=0'
)
ntm.fit({"train": train_input})

### 12. Deploy NTM and Generate Predictions

Deploy the trained NTM model and generate topic weight predictions.
Note that NTM returns `topic_weights` instead of `topic_mixture`.

In [ ]:
ntm_endpoint = ntm.deploy(
    initial_instance_count=1,
    instance_type="ml.m4.xlarge",
)
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import JSONDeserializer
ntm_endpoint.serializer = CSVSerializer()
ntm_endpoint.deserializer = JSONDeserializer()
results = ntm_endpoint.predict(lda_df[:10].to_numpy())
results_df = pd.DataFrame(results['predictions'])
results_df = results_df['topic_weights'].astype(str).str.split(',', expand=True)
results_df.columns = ['Topic1', 'Topic2', 'Topic3', 'Topic4', 'Topic5']
results_df['Topic1'] = results_df['Topic1'].apply(lambda x: remove_brackets(x))
results_df['Topic5'] = results_df['Topic5'].apply(lambda x: remove_brackets(x))
results_df = results_df.apply(pd.to_numeric, axis=1)
results_df

### 13. Cleanup NTM Endpoint

Delete the NTM inference endpoint to avoid ongoing charges.

In [ ]:
ntm_endpoint.delete_endpoint()
print("NTM endpoint deleted.")

---
## Summary

In this chapter we explored two approaches to topic modeling:

| Approach | Key Characteristics |
|----------|-------------------|
| **SageMaker LDA** | Classical probabilistic model, interpretable alpha/beta parameters |
| **SageMaker NTM** | Neural network-based, can capture complex patterns |

Each approach has its strengths: LDA offers interpretability, and NTM provides flexibility. In topic-modeling-comprehend.ipynb, we will look at topic modeling using Amazon Comprehend.